# Experiment: Ablation Study — Data & RAG Setup

**Ziel:** Datenbasis für die Ablation Study zum Monolith-RAG aufbauen.  
**Unternehmen:** AAPL, MSFT, AMZN, META  
**Fiscal Years:** FY2022 + FY2024  
**Filings:** 8 × 10-K

---

## 1. Setup

In [2]:
import logging
from src.common import download_filing, load_processed_filing, setup_logging

setup_logging(logging.INFO)

## 2. Download: 8 × 10-K Filings

Retry-Logik ist jetzt direkt in `ingestion.py` via `tenacity` eingebaut.  
Download-Kaskade: edgartools → datamule/secsgml → httpx direkt.

| Ticker | FY2022 | FY2024 |
|--------|--------|--------|
| AAPL   | ⬜     | ⬜     |
| MSFT   | ⬜     | ⬜     |
| AMZN   | ⬜     | ⬜     |
| META   | ⬜     | ⬜     |

In [3]:
# Define ablation study dataset
ABLATION_FILINGS = [
    ("AAPL",  2022),
    ("AAPL",  2023),
    ("AAPL",  2024),
    ("MSFT",  2022),
    ("MSFT",  2023),
    ("MSFT",  2024),
    ("AMZN",  2022),
    ("AMZN",  2023),
    ("AMZN",  2024),
    ("GOOGL",  2022),
    ("GOOGL",  2023),
    ("GOOGL",  2024),
]

In [4]:
# Download all filings
# Retry-Logik (tenacity) ist in ingestion.py eingebaut — kein manuelles Retry nötig.

results = {}
failed = []

for i, (ticker, fy) in enumerate(ABLATION_FILINGS):
    key = f"{ticker}_FY{fy}"
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(ABLATION_FILINGS)}] Downloading {key}...")
    print(f"{'='*60}")

    try:
        filing = download_filing(ticker, fiscal_year=fy)
        results[key] = filing
        print(f"  ✓ {key}: {len(filing.sections)} sections, {len(filing.full_text):,} chars")
    except Exception as e:
        failed.append(key)
        print(f"  ✗ {key}: FAILED — {e}")

print(f"\n\n{'='*60}")
print(f"Downloaded {len(results)}/{len(ABLATION_FILINGS)} filings")
if failed:
    print(f"Failed: {', '.join(failed)}")

2026-03-23 13:06:20 [INFO] src.common.ingestion: ⏭ Skipping download — AAPL FY2022 already exists: 10K_2022-10-28.md
2026-03-23 13:06:20 [INFO] src.common.ingestion: ⏭ Skipping download — AAPL FY2023 already exists: 10K_2023-11-03.md
2026-03-23 13:06:20 [INFO] src.common.ingestion: ⏭ Skipping download — AAPL FY2024 already exists: 10K_2024-11-01.md
2026-03-23 13:06:20 [INFO] src.common.ingestion: ⏭ Skipping download — MSFT FY2022 already exists: 10K_2022-07-28.md
2026-03-23 13:06:20 [INFO] src.common.ingestion: ⏭ Skipping download — MSFT FY2023 already exists: 10K_2023-07-27.md
2026-03-23 13:06:20 [INFO] src.common.ingestion: ⏭ Skipping download — MSFT FY2024 already exists: 10K_2024-07-30.md
2026-03-23 13:06:20 [INFO] src.common.ingestion: ⏭ Skipping download — AMZN FY2022 already exists: 10K_2023-02-03.md
2026-03-23 13:06:20 [INFO] src.common.ingestion: ⏭ Skipping download — AMZN FY2023 already exists: 10K_2024-02-02.md
2026-03-23 13:06:20 [INFO] src.common.ingestion: ⏭ Skipping down


[1/12] Downloading AAPL_FY2022...
  ✓ AAPL_FY2022: 5 sections, 213,033 chars

[2/12] Downloading AAPL_FY2023...
  ✓ AAPL_FY2023: 5 sections, 200,279 chars

[3/12] Downloading AAPL_FY2024...
  ✓ AAPL_FY2024: 5 sections, 200,703 chars

[4/12] Downloading MSFT_FY2022...
  ✓ MSFT_FY2022: 5 sections, 366,696 chars

[5/12] Downloading MSFT_FY2023...
  ✓ MSFT_FY2023: 5 sections, 202,007 chars

[6/12] Downloading MSFT_FY2024...
  ✓ MSFT_FY2024: 5 sections, 201,364 chars

[7/12] Downloading AMZN_FY2022...
  ✓ AMZN_FY2022: 5 sections, 276,861 chars

[8/12] Downloading AMZN_FY2023...
  ✓ AMZN_FY2023: 5 sections, 286,695 chars

[9/12] Downloading AMZN_FY2024...
  ✓ AMZN_FY2024: 5 sections, 283,492 chars

[10/12] Downloading GOOGL_FY2022...
  ✓ GOOGL_FY2022: 1 sections, 1,980 chars

[11/12] Downloading GOOGL_FY2023...
  ✓ GOOGL_FY2023: 5 sections, 364,155 chars

[12/12] Downloading GOOGL_FY2024...
  ✓ GOOGL_FY2024: 5 sections, 178,991 chars


Downloaded 12/12 filings


## 3. Übersicht: Heruntergeladene Filings

In [5]:
# Summary table
print(f"{'Key':<15} | {'Company':<30} | {'Filed':<12} | {'FY End':<12} | {'Sections':>8} | {'Chars':>10}")
print("-" * 95)
for key, f in results.items():
    print(
        f"{key:<15} | {f.metadata.company_name:<30} | "
        f"{f.metadata.filing_date:<12} | {f.metadata.fiscal_year_end:<12} | "
        f"{len(f.sections):>8} | {len(f.full_text):>10,}"
    )

Key             | Company                        | Filed        | FY End       | Sections |      Chars
-----------------------------------------------------------------------------------------------
AAPL_FY2022     | Apple Inc.                     | 2022-10-28   | 2022-09-24   |        5 |    213,033
AAPL_FY2023     | Apple Inc.                     | 2023-11-03   | 2023-09-30   |        5 |    200,279
AAPL_FY2024     | Apple Inc.                     | 2024-11-01   | 2024-09-28   |        5 |    200,703
MSFT_FY2022     | MICROSOFT CORP                 | 2022-07-28   | 2022-06-30   |        5 |    366,696
MSFT_FY2023     | MICROSOFT CORP                 | 2023-07-27   | 2023-06-30   |        5 |    202,007
MSFT_FY2024     | MICROSOFT CORP                 | 2024-07-30   | 2024-06-30   |        5 |    201,364
AMZN_FY2022     | AMAZON COM INC                 | 2023-02-03   | 2022-12-31   |        5 |    276,861
AMZN_FY2023     | AMAZON COM INC                 | 2024-02-02   | 2023-12-31   |

In [6]:
# Section breakdown for one filing (spot check)
sample = list(results.values())[0] if results else None
if sample:
    print(f"\nSections in {sample.metadata.ticker} FY{sample.metadata.fiscal_year_end[:4]}:")
    print("-" * 60)
    for section_name, content in sample.sections.items():
        print(f"  {section_name:45s} | {len(content):>8,} chars")


Sections in AAPL FY2022:
------------------------------------------------------------
  Business                                      |   14,709 chars
  Risk Factors                                  |   70,911 chars
  MD&A                                          |   17,594 chars
  Directors and Corporate Governance            |      405 chars
  Financial Statements                          |  109,117 chars


## 4. Validierung: Datenqualität prüfen

In [7]:
# Validate: each filing should have at least 3 key sections
REQUIRED_SECTIONS = {"Business", "Risk Factors", "MD&A"}

print(f"{'Key':<15} | {'Sections':>8} | {'Missing':>40}")
print("-" * 70)

all_ok = True
for key, f in results.items():
    present = set(f.sections.keys())
    missing = REQUIRED_SECTIONS - present
    status = '✓' if not missing else '⚠'
    missing_str = ', '.join(missing) if missing else '—'
    print(f"  {status} {key:<13} | {len(f.sections):>8} | {missing_str:>40}")
    if missing:
        all_ok = False

print()
if all_ok:
    print("✅ Alle Filings haben die 3 Kern-Sections.")
else:
    print("⚠️  Einige Filings haben fehlende Sections — prüfe die Logs oben.")

Key             | Sections |                                  Missing
----------------------------------------------------------------------
  ✓ AAPL_FY2022   |        5 |                                        —
  ✓ AAPL_FY2023   |        5 |                                        —
  ✓ AAPL_FY2024   |        5 |                                        —
  ✓ MSFT_FY2022   |        5 |                                        —
  ✓ MSFT_FY2023   |        5 |                                        —
  ✓ MSFT_FY2024   |        5 |                                        —
  ✓ AMZN_FY2022   |        5 |                                        —
  ✓ AMZN_FY2023   |        5 |                                        —
  ✓ AMZN_FY2024   |        5 |                                        —
  ⚠ GOOGL_FY2022  |        1 |             Risk Factors, MD&A, Business
  ✓ GOOGL_FY2023  |        5 |                                        —
  ✓ GOOGL_FY2024  |        5 |                                     